# Letterboxd Friends Taste Analysis

This notebook compares movie taste across the Letterboxd exports in this folder.

Scope:

- Use only `ratings.csv` and `reviews.csv`.
- Ignore watched-only history, likes, watchlists, lists, comments, deleted exports, and orphaned exports.
- Match films by normalized `Name + Year`, not by Letterboxd URI.
- Reviewed-but-unrated films count for user activity only; rating comparisons use films with ratings.

In [1]:
from pathlib import Path
from itertools import combinations
import re

import numpy as np
import pandas as pd

ROOT = Path(".")
RATING_BUCKETS = [x / 2 for x in range(1, 11)]
MIN_GROUP_RATERS = 3

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.options.display.float_format = "{:.2f}".format


def username_from_export(path):
    match = re.match(r"letterboxd-(.*)-\d{4}-\d{2}-\d{2}-.*-utc$", path.name)
    return match.group(1) if match else path.name


def normalize_title(value):
    value = str(value or "").strip().lower()
    value = value.replace("&", "and")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def add_film_key(df):
    df = df.copy()
    df["Name"] = df["Name"].fillna("").astype(str)
    df["Year"] = df["Year"].fillna("").astype(str).str.replace(r"\.0$", "", regex=True)
    df["title_norm"] = df["Name"].map(normalize_title)
    df["film_key"] = df["title_norm"] + " (" + df["Year"].astype(str).str.strip() + ")"
    df["film"] = df["Name"] + " (" + df["Year"].astype(str).str.strip() + ")"
    return df


def read_export_csv(path):
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def pct(series):
    return 100 * series.mean() if len(series) else np.nan

## Load Ratings And Reviews

Ratings come from `ratings.csv` plus review rows that include a rating. If the same user has a rating in both places, `ratings.csv` is preferred.

In [2]:
exports = sorted(path for path in ROOT.glob("letterboxd-*-utc") if path.is_dir())

rating_frames = []
review_frames = []

for export in exports:
    user = username_from_export(export)

    ratings = read_export_csv(export / "ratings.csv")
    if not ratings.empty:
        ratings = add_film_key(ratings)
        ratings["user"] = user
        ratings["source"] = "ratings.csv"
        ratings["Rating"] = pd.to_numeric(ratings["Rating"], errors="coerce")
        rating_frames.append(ratings[["user", "film_key", "film", "Name", "Year", "Rating", "source"]])

    reviews = read_export_csv(export / "reviews.csv")
    if not reviews.empty:
        reviews = add_film_key(reviews)
        reviews["user"] = user
        reviews["source"] = "reviews.csv"
        reviews["Rating"] = pd.to_numeric(reviews.get("Rating"), errors="coerce")
        review_frames.append(reviews[["user", "film_key", "film", "Name", "Year", "Rating", "Review", "source"]])

raw_ratings = pd.concat(rating_frames, ignore_index=True) if rating_frames else pd.DataFrame()
raw_reviews = pd.concat(review_frames, ignore_index=True) if review_frames else pd.DataFrame()

# Prefer explicit ratings.csv values when duplicate user-film ratings exist.
review_ratings = raw_reviews.dropna(subset=["Rating"])[["user", "film_key", "film", "Name", "Year", "Rating", "source"]]
all_rating_sources = pd.concat([raw_ratings, review_ratings], ignore_index=True)
all_rating_sources["source_priority"] = all_rating_sources["source"].map({"ratings.csv": 0, "reviews.csv": 1})

ratings = (
    all_rating_sources
    .sort_values(["user", "film_key", "source_priority"])
    .drop_duplicates(["user", "film_key"], keep="first")
    .drop(columns=["source_priority"])
    .reset_index(drop=True)
)

reviewed_films = (
    raw_reviews[["user", "film_key", "film", "Name", "Year"]]
    .drop_duplicates(["user", "film_key"])
    .reset_index(drop=True)
)

considered_films = (
    pd.concat(
        [
            ratings[["user", "film_key", "film", "Name", "Year"]],
            reviewed_films[["user", "film_key", "film", "Name", "Year"]],
        ],
        ignore_index=True,
    )
    .drop_duplicates(["user", "film_key"])
    .reset_index(drop=True)
)

users = sorted(considered_films["user"].unique())
print(f"Loaded {len(users)} users: {', '.join(users)}")
print(f"Rated user-film rows: {len(ratings)}")
print(f"Reviewed user-film rows: {len(reviewed_films)}")
print(f"Considered user-film rows: {len(considered_films)}")

Loaded 4 users: beap2, bigguccisosa777, faustozampa, will22
Rated user-film rows: 975
Reviewed user-film rows: 84
Considered user-film rows: 1009


## User Summary

Bucket columns show the percentage of each user's ratings that fall into each exact Letterboxd rating value.

In [3]:
summary = (
    considered_films.groupby("user")["film_key"].nunique()
    .rename("films considered")
    .to_frame()
    .join(ratings.groupby("user")["film_key"].nunique().rename("rated films"))
    .join(reviewed_films.groupby("user")["film_key"].nunique().rename("reviewed films"))
    .join(ratings.groupby("user")["Rating"].mean().rename("average rating"))
    .join(ratings.groupby("user")["Rating"].median().rename("median rating"))
)

rating_counts = pd.crosstab(ratings["user"], ratings["Rating"])
rating_percentages = rating_counts.div(rating_counts.sum(axis=1), axis=0).mul(100)

for bucket in RATING_BUCKETS:
    if bucket not in rating_percentages:
        rating_percentages[bucket] = 0.0

rating_percentages = (
    rating_percentages[RATING_BUCKETS]
    .rename(columns={bucket: f"% {bucket:.1f}" for bucket in RATING_BUCKETS})
)

user_summary = (
    summary.join(rating_percentages)
    .fillna({"rated films": 0, "reviewed films": 0})
    .sort_index()
)

user_summary

,films considered,rated films,reviewed films,average rating,median rating,% 0.5,% 1.0,% 1.5,% 2.0,% 2.5,% 3.0,% 3.5,% 4.0,% 4.5,% 5.0
user,,,,,,,,,,,,,,,
beap2,204,204,7,3.48,3.50,0.00,2.45,0.98,8.82,5.39,22.55,13.24,27.94,12.75,5.88
bigguccisosa777,402,402,3,3.39,3.50,1.24,1.99,3.48,5.22,9.45,19.40,21.64,19.65,9.20,8.71
faustozampa,130,128,26,3.50,3.50,0.78,3.91,4.69,5.47,9.38,11.72,17.19,17.97,14.84,14.06
will22,273,241,48,3.96,4.00,0.83,1.24,1.24,2.90,4.56,7.88,16.18,18.67,24.48,21.99


## Rating Personality

These labels summarize rating behavior, not taste quality.

In [4]:
personality = ratings.groupby("user").agg(
    average_rating=("Rating", "mean"),
    rating_std=("Rating", "std"),
    favorite_tier_pct=("Rating", lambda s: pct(s >= 4.5)),
    low_rating_pct=("Rating", lambda s: pct(s <= 2.5)),
    middle_rating_pct=("Rating", lambda s: pct(s.isin([3.0, 3.5]))),
)

personality_signals = pd.DataFrame(
    [
        ("most generous", personality.sort_values(["average_rating", "favorite_tier_pct"], ascending=False).index[0]),
        ("strictest", personality.sort_values(["low_rating_pct", "average_rating"], ascending=[False, True]).index[0]),
        ("most polarized", personality.sort_values("rating_std", ascending=False).index[0]),
        ("most moderate", personality.sort_values("rating_std", ascending=True).index[0]),
    ],
    columns=["signal", "user"],
).join(personality, on="user")

personality_signals

,signal,user,average_rating,rating_std,favorite_tier_pct,low_rating_pct,middle_rating_pct
0,most generous,will22,3.96,0.96,46.47,10.79,24.07
1,strictest,faustozampa,3.50,1.13,28.91,24.22,28.91
2,most polarized,faustozampa,3.50,1.13,28.91,24.22,28.91
3,most moderate,beap2,3.48,0.91,18.63,17.65,35.78


## Pairwise Similarity

Only films rated by both users are used here. The table is sorted from most similar to most dissimilar by mean absolute rating difference.

In [5]:
pairwise_rows = []
pairwise_detail_frames = {}

for user_a, user_b in combinations(users, 2):
    a = ratings[ratings["user"] == user_a][["film_key", "film", "Rating"]].rename(columns={"Rating": user_a})
    b = ratings[ratings["user"] == user_b][["film_key", "Rating"]].rename(columns={"Rating": user_b})
    common = a.merge(b, on="film_key", how="inner")
    common["difference"] = (common[user_a] - common[user_b]).abs()
    common = common.sort_values(["difference", "film"], ascending=[False, True]).reset_index(drop=True)
    pairwise_detail_frames[(user_a, user_b)] = common

    pairwise_rows.append(
        {
            "pair": f"{user_a} / {user_b}",
            "common rated films": len(common),
            "mean abs diff": common["difference"].mean(),
            "median abs diff": common["difference"].median(),
            "% exactly equal": pct(common["difference"] == 0),
            "% within 0.5": pct(common["difference"] <= 0.5),
            "strong disagreements": int((common["difference"] >= 2.0).sum()),
        }
    )

pairwise_similarity = (
    pd.DataFrame(pairwise_rows)
    .sort_values("mean abs diff", ascending=True)
    .reset_index(drop=True)
)

pairwise_similarity

,pair,common rated films,mean abs diff,median abs diff,% exactly equal,% within 0.5,strong disagreements
0,bigguccisosa777 / will22,80,0.57,0.50,31.25,71.25,2
1,beap2 / faustozampa,65,0.69,0.50,23.08,61.54,5
2,beap2 / will22,59,0.71,0.50,20.34,52.54,3
3,beap2 / bigguccisosa777,59,0.75,0.50,16.95,52.54,4
4,bigguccisosa777 / faustozampa,46,1.01,0.50,19.57,54.35,9
5,faustozampa / will22,39,1.27,1.00,12.82,38.46,10


## Pairwise Detail Tables

For each pair, inspect the biggest disagreements, exact matches, shared favorites, and shared dislikes.

In [6]:
for user_a, user_b in combinations(users, 2):
    common = pairwise_detail_frames[(user_a, user_b)]
    print(f"\n\n=== {user_a} / {user_b} ===")

    print("\nBiggest disagreements")
    display(common[["film", user_a, user_b, "difference"]].head(10))

    print("\nExact same ratings")
    display(common.loc[common["difference"] == 0, ["film", user_a, user_b, "difference"]].sort_values("film").head(10))

    print("\nShared favorites")
    display(
        common.loc[(common[user_a] >= 4.5) & (common[user_b] >= 4.5), ["film", user_a, user_b, "difference"]]
        .sort_values(["difference", "film"])
        .head(10)
    )

    print("\nShared dislikes")
    display(
        common.loc[(common[user_a] <= 2.5) & (common[user_b] <= 2.5), ["film", user_a, user_b, "difference"]]
        .sort_values(["difference", "film"])
        .head(10)
    )



=== beap2 / bigguccisosa777 ===

Biggest disagreements


,film,beap2,bigguccisosa777,difference
0,American Beauty (1999),4.50,2.50,2.00
1,Django Unchained (2012),3.00,5.00,2.00
2,Dune: Part Two (2024),5.00,3.00,2.00
3,Project Hail Mary (2026),2.50,4.50,2.00
4,Ex Machina (2015),4.00,2.50,1.50
5,La La Land (2016),5.00,3.50,1.50
6,The Grand Budapest Hotel (2014),3.50,5.00,1.50
7,The King's Speech (2010),3.50,2.00,1.50
8,2001: A Space Odyssey (1968),4.00,5.00,1.00
9,Anora (2024),4.00,3.00,1.00



Exact same ratings


,film,beap2,bigguccisosa777,difference
49,Arrival (2016),3.00,3.00,0.00
50,Il Divo (2008),3.50,3.50,0.00
51,Il Sorpasso (1962),4.50,4.50,0.00
52,In the Mood for Love (2000),4.50,4.50,0.00
53,Marty Supreme (2025),4.50,4.50,0.00
54,No Other Choice (2025),4.00,4.00,0.00
55,One Battle After Another (2025),4.50,4.50,0.00
56,Parthenope (2024),1.00,1.00,0.00
57,The Hand of God (2021),3.00,3.00,0.00
58,Up (2009),4.00,4.00,0.00



Shared favorites


...

,film,faustozampa,will22,difference
34,In the Mood for Love (2000),4.50,4.50,0.00
35,Inception (2010),4.00,4.00,0.00
36,Kill Bill: Vol. 1 (2003),3.00,3.00,0.00
37,No Country for Old Men (2007),5.00,5.00,0.00
38,Parasite (2019),5.00,5.00,0.00



Shared favorites


,film,faustozampa,will22,difference
34,In the Mood for Love (2000),4.50,4.50,0.00
37,No Country for Old Men (2007),5.00,5.00,0.00
38,Parasite (2019),5.00,5.00,0.00
25,La La Land (2016),5.00,4.50,0.50
33,The Truman Show (1998),4.50,5.00,0.50



Shared dislikes


,film,faustozampa,will22,difference


## Group Metrics

Group tables use films rated by at least 3 users.

In [7]:
rating_matrix = ratings.pivot_table(index="film", columns="user", values="Rating", aggfunc="first")

group_metrics = pd.DataFrame(
    {
        "raters": rating_matrix.notna().sum(axis=1),
        "average": rating_matrix.mean(axis=1),
        "spread": rating_matrix.max(axis=1) - rating_matrix.min(axis=1),
        "std dev": rating_matrix.std(axis=1, ddof=0),
    }
)

group_metrics = (
    group_metrics[group_metrics["raters"] >= MIN_GROUP_RATERS]
    .join(rating_matrix)
    .sort_values(["raters", "film"], ascending=[False, True])
)

print(f"Group-qualified films: {len(group_metrics)}")
group_metrics.head(50)

Group-qualified films: 63


,raters,average,spread,std dev,beap2,bigguccisosa777,faustozampa,will22
film,,,,,,,,
2001: A Space Odyssey (1968),4,4.00,3.00,1.22,4.00,5.00,2.00,5.00
Barry Lyndon (1975),4,3.50,2.00,0.79,3.00,4.00,2.50,4.50
Blade Runner (1982),4,4.50,1.50,0.61,4.50,5.00,3.50,5.00
Challengers (2024),4,3.50,1.50,0.61,4.00,3.50,4.00,2.50
Decision to Leave (2022),4,4.38,1.00,0.41,4.00,4.50,4.00,5.00
In the Mood for Love (2000),4,4.50,0.00,0.00,4.50,4.50,4.50,4.50
La La Land (2016),4,4.50,1.50,0.61,5.00,3.50,5.00,4.50
Mulholland Drive (2001),4,4.12,1.50,0.65,4.50,3.50,3.50,5.00
Oldboy (2003),4,4.25,0.50,0.25,4.50,4.00,4.00,4.50


## Consensus And Divisiveness

Consensus favorites require average rating `>= 4.0` and spread `<= 1.0`.

Consensus dislikes require average rating `<= 3.0` and spread `<= 1.0`.

Divisive films are sorted by spread, then standard deviation.

In [8]:
consensus_favorites = (
    group_metrics[(group_metrics["average"] >= 4.0) & (group_metrics["spread"] <= 1.0)]
    .sort_values(["average", "spread"], ascending=[False, True])
)

consensus_dislikes = (
    group_metrics[(group_metrics["average"] <= 3.0) & (group_metrics["spread"] <= 1.0)]
    .sort_values(["average", "spread"], ascending=[True, True])
)

most_divisive = group_metrics.sort_values(["spread", "std dev"], ascending=[False, False])

print("Consensus favorites")
display(consensus_favorites.head(20))

print("Consensus dislikes")
display(consensus_dislikes.head(20))

print("Most divisive films")
display(most_divisive.head(20))

Consensus favorites


,raters,average,spread,std dev,beap2,bigguccisosa777,faustozampa,will22
film,,,,,,,,
No Country for Old Men (2007),3,4.83,0.50,0.24,NaN,4.50,5.00,5.00
Parasite (2019),3,4.83,0.50,0.24,4.50,NaN,5.00,5.00
Forrest Gump (1994),3,4.67,1.00,0.47,5.00,NaN,5.00,4.00
GoodFellas (1990),3,4.67,1.00,0.47,4.00,5.00,NaN,5.00
Prisoners (2013),3,4.67,1.00,0.47,5.00,4.00,5.00,NaN
Se7en (1995),3,4.67,1.00,0.47,NaN,5.00,5.00,4.00
The Social Network (2010),3,4.67,1.00,0.47,5.00,NaN,5.00,4.00
We All Loved Each Other So Much (1974),3,4.67,1.00,0.47,4.00,5.00,NaN,5.00
In the Mood for Love (2000),4,4.50,0.00,0.00,4.50,4.50,4.50,4.50


Consensus dislikes


,raters,average,spread,std dev,beap2,bigguccisosa777,faustozampa,will22
film,,,,,,,,
Barbie (2023),3,1.83,0.50,0.24,2.00,1.50,NaN,2.00
Arrival (2016),3,2.83,0.50,0.24,3.00,3.00,2.50,NaN


Most divisive films


,raters,average,spread,std dev,beap2,bigguccisosa777,faustozampa,will22
film,,,,,,,,
Mystic River (2003),3,3.17,4.00,1.65,NaN,3.50,1.00,5.00
Project Hail Mary (2026),3,2.83,3.00,1.25,2.50,4.50,1.50,NaN
2001: A Space Odyssey (1968),4,4.00,3.00,1.22,4.00,5.00,2.00,5.00
The Substance (2024),3,3.67,2.50,1.18,NaN,4.50,2.00,4.50
Vertigo (1958),3,4.17,2.50,1.18,NaN,5.00,2.50,5.00
American Beauty (1999),3,3.00,2.50,1.08,4.50,2.50,2.00,NaN
Eyes Wide Shut (1999),3,4.00,2.50,1.08,4.50,5.00,2.50,NaN
Nosferatu (2024),3,3.00,2.50,1.08,NaN,4.00,1.50,3.50
Ex Machina (2015),3,3.83,2.50,1.03,4.00,2.50,5.00,NaN


## One Person Against The Group

A user is an outlier when their rating differs from the average rating of the other users by at least `1.5` stars.

In [9]:
outlier_rows = []

for film, row in rating_matrix.loc[group_metrics.index].iterrows():
    user_ratings = row.dropna()
    if len(user_ratings) < MIN_GROUP_RATERS:
        continue

    for user, user_rating in user_ratings.items():
        others = user_ratings.drop(index=user)
        if len(others) < 2:
            continue
        others_average = others.mean()
        difference = user_rating - others_average
        if abs(difference) >= 1.5:
            outlier_rows.append(
                {
                    "film": film,
                    "outlier": user,
                    "direction": "higher" if difference > 0 else "lower",
                    "outlier rating": user_rating,
                    "others average": others_average,
                    "difference": difference,
                }
            )

outliers = (
    pd.DataFrame(outlier_rows)
    .assign(abs_difference=lambda df: df["difference"].abs() if len(df) else pd.Series(dtype=float))
    .sort_values(["abs_difference", "film", "outlier"], ascending=[False, True, True])
    .drop(columns=["abs_difference"], errors="ignore")
    .reset_index(drop=True)
)

print(f"Outlier cases: {len(outliers)}")
outliers.head(30)

Outlier cases: 23


,film,outlier,direction,outlier rating,others average,difference
0,Mystic River (2003),faustozampa,lower,1.00,4.25,-3.25
1,Mystic River (2003),will22,higher,5.00,2.25,2.75
2,2001: A Space Odyssey (1968),faustozampa,lower,2.00,4.67,-2.67
3,Project Hail Mary (2026),bigguccisosa777,higher,4.50,2.00,2.50
4,The Substance (2024),faustozampa,lower,2.00,4.50,-2.50
5,Vertigo (1958),faustozampa,lower,2.50,5.00,-2.50
6,American Beauty (1999),beap2,higher,4.50,2.25,2.25
7,Eyes Wide Shut (1999),faustozampa,lower,2.50,4.75,-2.25
8,Nosferatu (2024),faustozampa,lower,1.50,3.75,-2.25
9,Ex Machina (2015),bigguccisosa777,lower,2.50,4.50,-2.00


## Thresholds To Tweak Later

Useful values are defined near the top of the notebook:

- `MIN_GROUP_RATERS = 3`
- consensus favorite: average `>= 4.0`, spread `<= 1.0`
- consensus dislike: average `<= 3.0`, spread `<= 1.0`
- outlier: absolute difference from the other users' average `>= 1.5`